# Tutorial: Mastering Genomics with DeepChem
### *From Raw DNA to State-of-the-Art Foundation Models*

This tutorial demonstrates how to use the `genomics` module to process, featurize, and model genomic sequences using DeepChem's standard MolNet architecture.

We will use the **Human Non-TATA Promoters** dataset from Genomic Benchmarks to walk through a complete research workflow.

## Colab

This tutorial can be run in Google Colab. Upload this notebook or clone the repository, then run the Setup cell to install dependencies. For similar DeepChem tutorials (e.g. ProtBERT, ChemBERTa), see the [DeepChem tutorials](https://github.com/deepchem/deepchem/tree/master/examples/tutorials).


## Setup

Install DeepChem (with PyTorch), transformers, and genomic-benchmarks. Skip this cell if you already have a suitable environment.

In [ ]:
!pip install -q "deepchem[torch]" transformers genomic-benchmarks scikit-learn matplotlib
import deepchem as dc
print(f"DeepChem {dc.__version__}")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from genomics.loader import load_genomic_benchmark
from genomics.featurizers import DNAKmerCountFeaturizer, DNAOneHotFeaturizer
from sklearn.ensemble import RandomForestClassifier

np.random.seed(42)


## Section 1: Standardized Data Loading
DeepChem's `load_genomic_benchmark` function handles downloading, splitting, and memory-efficient storage. 

We use `splitter="official"` to ensure we use the benchmark's standardized Train/Test sets, which is critical for scientific reproducibility.


In [ ]:
# Load the dataset using the official benchmark splits
tasks, datasets, transformers = load_genomic_benchmark(
    dataset_name="human_nontata_promoters",
    featurizer=None,    # Load raw sequences first
    splitter="official",
    reload=False
)

# Some benchmarks might only have Train/Test (Valid may be empty)
train, test = datasets[0], datasets[-1]

print(f"Tasks: {tasks}")
print(f"Train samples: {len(train)}")
print(f"Test samples:  {len(test)}")
print(f"\nSample DNA sequence: {train.ids[0][:50]}...")


## Section 2: Flexible DNA Featurization
Genomic sequences need to be converted into numerical tensors. DeepChem makes this effortless.

1. **One-Hot Encoding**: Best for CNNs.
2. **K-mer Frequency**: Best for traditional ML and Transformers.


In [ ]:
# 1. One-Hot Featurization (Shortcut: "onehot")
_, oh_datasets, _ = load_genomic_benchmark(
    dataset_name="human_nontata_promoters",
    featurizer="onehot",
    splitter="official",
    reload=False
)
oh_train = oh_datasets[0]
print(f"One-Hot Shape: {oh_train.X.shape} (N, Length, 4)")

# 2. K-mer Featurization (Using 4-mers)
kmer_feat = DNAKmerCountFeaturizer(k=4)
_, kmer_datasets, _ = load_genomic_benchmark(
    dataset_name="human_nontata_promoters",
    featurizer=kmer_feat,
    splitter="official",
    reload=False
)
kmer_train = kmer_datasets[0]
print(f"K-mer Shape: {kmer_train.X.shape} (N, 256 frequencies)")


## Section 3: Data Visualization
Understanding the "signal" in your genomic data. Let's visualize the 4-mer frequency distribution between promoters and non-promoters.


In [ ]:
# Compare mean K-mer frequencies across classes
pos_mask = (kmer_train.y == 1).flatten()
neg_mask = (kmer_train.y == 0).flatten()

pos_mean = kmer_train.X[pos_mask].mean(axis=0)
neg_mean = kmer_train.X[neg_mask].mean(axis=0)

plt.figure(figsize=(12, 4))
plt.plot(pos_mean, label="Promoter (Positive)", alpha=0.8)
plt.plot(neg_mean, label="Non-Promoter (Negative)", alpha=0.8)
plt.fill_between(range(256), pos_mean, neg_mean, color='gray', alpha=0.2)
plt.title("4-mer Frequency Signature: Promoters vs Non-Promoters")
plt.xlabel("K-mer Index")
plt.ylabel("Mean Frequency")
plt.legend()
plt.show()


## Section 4: Baseline Modeling with Random Forest
We use DeepChem's `SklearnModel` wrapper to train a quick baseline on our K-mer features.


In [ ]:
# Initialize and train
rf = RandomForestClassifier(n_estimators=100, n_jobs=-1)
model = dc.models.SklearnModel(rf)
model.fit(kmer_train)

# Evaluate using DeepChem's ROC-AUC metric (same as ProtBERT/ChemBERTa tutorials)
metric = dc.metrics.Metric(dc.metrics.roc_auc_score, mode="classification")
kmer_test = kmer_datasets[-1]
test_score = model.evaluate(kmer_test, [metric])

print(f"Baseline Test ROC-AUC: {test_score['roc_auc_score']:.4f}")


## Section 5: Foundation Models (DNABERT-2)
Finally, we scale up to a Transformer-based foundation model. DNABERT-2 is pre-trained on massive genomic corpora and can be fine-tuned for specific tasks.


In [ ]:
from genomics.dnabert2 import DNABERT2Model

# Same pattern as ProtBERT/ChemBERTa: model_dir for checkpoints
model_dir = os.path.join(os.getcwd(), "dnabert2_checkpoints")
os.makedirs(model_dir, exist_ok=True)

bert_model = DNABERT2Model(
    task="classification",
    n_tasks=1,
    model_dir=model_dir,
    batch_size=4,
    max_seq_length=128,
    learning_rate=2e-5,
)

bert_model.fit(train, nb_epoch=1)
bert_score = bert_model.evaluate(test, [metric], n_classes=2)
print(f"DNABERT-2 Test ROC-AUC: {bert_score[metric.name]:.4f}")


## References

- **Genomic Benchmarks**: [ML-Bioinfo-CEITEC/genomic_benchmarks](https://github.com/ML-Bioinfo-CEITEC/genomic_benchmarks)
- **DNABERT-2**: Zhou, Z. et al. DNABERT-2: Efficient Foundation Model and Benchmark for Multi-Species Genome. arXiv:2306.15006 (2023)
- **DeepChem**: [deepchem/deepchem](https://github.com/deepchem/deepchem)